# Block 2 Solution — Generate a Minimal Define-XML v2.1

Completed version of `02_create_define/create_define_exercise.ipynb`. The markdown notes at
each TODO explain the approach.

In [ ]:
import os
import odmlib.define_2_1.model as DEF

os.makedirs("output", exist_ok=True)

In [ ]:
odm = DEF.ODM(
    FileOID="DEF.RPH2026.DM",
    FileType="Snapshot",
    CreationDateTime="2026-08-19T12:00:00",
    ODMVersion="1.3.2",
    Context="Submission",
    Originator="R/Pharma 2026 Workshop",
    SourceSystem="odmlib",
)

study = DEF.Study(OID="ST.RPH2026")
study.GlobalVariables = DEF.GlobalVariables(
    StudyName=DEF.StudyName(_content="RPH2026"),
    StudyDescription=DEF.StudyDescription(_content="R/Pharma 2026 odmlib workshop study"),
    ProtocolName=DEF.ProtocolName(_content="RPH-2026-001"),
)

print(study.GlobalVariables.StudyName)

In [ ]:
mdv = DEF.MetaDataVersion(
    OID="MDV.RPH2026.1",
    Name="RPH2026 Data Definitions",
    Description="Demographics metadata for the workshop",
    DefineVersion="2.1.0",
)

standards = DEF.Standards()
standards.Standard.append(
    DEF.Standard(OID="STD.1", Name="SDTMIG", Type="IG", Version="3.4", Status="Final"))
mdv.Standards = standards

print("MetaDataVersion ready:", mdv.OID)

## TODO 1 — The DM dataset and its variable references

Required attributes are enforced right here — drop `Repeating` and the constructor raises
`OdmlibRequiredAttributeError` immediately. Children are added in schema order: `Description`
first, then the `ItemRef`s (with `Class` and `leaf` following in TODO 2).

In [ ]:
igd = DEF.ItemGroupDef(
    OID="IG.DM", Name="DM", Repeating="No", IsReferenceData="No",
    SASDatasetName="DM", Domain="DM", Purpose="Tabulation",
    Structure="One record per subject",
    ArchiveLocationID="LF.DM",
    StandardOID="STD.1",
)

igd.Description = DEF.Description()
igd.Description.TranslatedText.append(DEF.TranslatedText(_content="Demographics", lang="en"))

igd.ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.STUDYID", Mandatory="Yes", OrderNumber=1, KeySequence=1))
igd.ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.USUBJID", Mandatory="Yes", OrderNumber=2, KeySequence=2))
igd.ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.AGE", Mandatory="No", OrderNumber=3))
igd.ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.SEX", Mandatory="Yes", OrderNumber=4))

print(f"{igd.Name}: {len(igd.ItemRef)} ItemRefs")

## TODO 2 — Class and leaf

`Class` and `leaf` are single-object children — assigned, not appended. The leaf's `ID`
matches the `ArchiveLocationID` from TODO 1; that cross-reference (like every other OID
reference) is checked by the OID checker below and in Block 3. `href` is an `xlink:href` and
the `def:title` is built like any text element — odmlib handles both namespaces.

In [ ]:
igd.Class = DEF.Class(Name="SPECIAL PURPOSE")
igd.leaf = DEF.leaf(ID="LF.DM", href="dm.xpt", title=DEF.title(_content="dm.xpt"))

print("Class:", igd.Class.Name, "| leaf:", igd.leaf.ID)

## TODO 3 — Variables and the SEX codelist

`make_item` encodes the schema-ordered child sequence for an `ItemDef` (`Description`,
`CodeListRef`, `Origin`) in one place — a pattern worth copying for real projects, where a
define.xml has hundreds of variables and consistency is everything.

In [ ]:
def make_item(oid, name, dtype, length, desc, codelist=None):
    item = DEF.ItemDef(OID=oid, Name=name, DataType=dtype, Length=length, SASFieldName=name)
    item.Description = DEF.Description()
    item.Description.TranslatedText.append(DEF.TranslatedText(_content=desc, lang="en"))
    if codelist:
        item.CodeListRef = DEF.CodeListRef(CodeListOID=codelist)
    item.Origin.append(DEF.Origin(Type="Collected"))
    return item

mdv.ItemDef.append(make_item("IT.DM.STUDYID", "STUDYID", "text", 12, "Study Identifier"))
mdv.ItemDef.append(make_item("IT.DM.USUBJID", "USUBJID", "text", 25, "Unique Subject Identifier"))
mdv.ItemDef.append(make_item("IT.DM.AGE", "AGE", "integer", 3, "Age"))
mdv.ItemDef.append(make_item("IT.DM.SEX", "SEX", "text", 1, "Sex", codelist="CL.SEX"))

cl = DEF.CodeList(OID="CL.SEX", Name="Sex", DataType="text")
for coded, decode in (("F", "Female"), ("M", "Male")):
    term = DEF.CodeListItem(CodedValue=coded)
    term.Decode = DEF.Decode()
    term.Decode.TranslatedText.append(DEF.TranslatedText(_content=decode, lang="en"))
    cl.CodeListItem.append(term)
mdv.CodeList.append(cl)

print(f"{len(mdv.ItemDef)} ItemDefs, {len(mdv.CodeList)} CodeLists")

## TODO 4 — Assemble and write

The one assembly gotcha: `Study` and `MetaDataVersion` are **single objects** in Define-XML
(one study, one metadata version per define.xml), so they are assigned — only the collections
use `append`.

In [ ]:
mdv.ItemGroupDef.append(igd)
study.MetaDataVersion = mdv      # assignment - single object
odm.Study = study                # assignment - single object

odm.write_xml("output/define_dm.xml")
print("wrote output/define_dm.xml")

In [ ]:
from odmlib import create_oid_checker

odm.verify_oids(create_oid_checker("define_2_1"))
print("All OID references resolve - your define.xml is internally consistent")

## Stretch — value-level metadata

`ValueListDef` + `WhereClauseDef` express "AGE, when AGEU = YEARS, is an integer in years".
Because these sections are added to an already-assembled document, they land out of schema
order in the `MetaDataVersion` — `verify_order()` reports it, and `reorder_object()` on the
flagged element repairs it (the emitted warning is odmlib telling you a reorder happened).
Note that `write_xml()` always serializes in schema order regardless; the order check matters
for `validate()` and for dict/JSON output.

In [ ]:
from odmlib.exceptions import OdmlibElementOrderError

# a normal AGEU variable
mdv.ItemDef.append(make_item("IT.DM.AGEU", "AGEU", "text", 6, "Age Units"))
igd.ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.AGEU", Mandatory="No", OrderNumber=5))

# the value-level item: AGE when it is recorded in years
mdv.ItemDef.append(make_item("IT.DM.AGE.YEARS", "AGE", "integer", 3, "Age in Years"))

# the condition: AGEU = "YEARS"
wcd = DEF.WhereClauseDef(OID="WC.DM.AGEU.YEARS")
check = DEF.RangeCheck(SoftHard="Soft", ItemOID="IT.DM.AGEU", Comparator="EQ")
check.CheckValue.append(DEF.CheckValue(_content="YEARS"))
wcd.RangeCheck.append(check)
mdv.WhereClauseDef.append(wcd)

# the value list: one value-level ItemRef guarded by the condition
vld = DEF.ValueListDef(OID="VL.DM.AGE")
vref = DEF.ItemRef(ItemOID="IT.DM.AGE.YEARS", Mandatory="No", OrderNumber=1)
vref.WhereClauseRef.append(DEF.WhereClauseRef(WhereClauseOID="WC.DM.AGEU.YEARS"))
vld.ItemRef.append(vref)
mdv.ValueListDef.append(vld)

# point AGE at its value-level metadata
mdv.find("ItemDef", "OID", "IT.DM.AGE").ValueListRef = DEF.ValueListRef(ValueListOID="VL.DM.AGE")

# late additions land out of schema order - verify, then repair
try:
    odm.verify_order()
except OdmlibElementOrderError as oe:
    print("order issue:", oe)
    mdv.reorder_object()
    odm.verify_order()
    print("fixed with reorder_object()")

odm.verify_oids(create_oid_checker("define_2_1"))
odm.write_xml("output/define_dm_vlm.xml")
print("wrote output/define_dm_vlm.xml")

Both files are ready for Block 3, where they go through all four validation layers —
including XSD schema validation against the bundled Define-XML v2.1 schema.